# Feature Engineering And Retraining

Retrain the selected top models after feature selection. This notebook uses validation data for model selection and keeps the test set for final evaluation.

In [4]:
from pathlib import Path
import time
import warnings

import joblib
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.svm import LinearSVC

warnings.filterwarnings("ignore")

ROOT_DIR = Path.cwd().parents[1] if Path.cwd().name == "feature_engineering" else Path.cwd()
DATA_DIR = ROOT_DIR / "data" / "classification"
OUTPUT_DIR = ROOT_DIR / "train" / "feature_engineering" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_COLUMN = "readmitted_binary"
RANDOM_STATE = 42

DATA_DIR, OUTPUT_DIR

(WindowsPath('d:/HocTap/KT&XLTT/CUOIKI/data/classification'),
 WindowsPath('d:/HocTap/KT&XLTT/CUOIKI/train/feature_engineering/outputs'))

## Load Data

In [5]:
train_df = pd.read_csv(DATA_DIR / "classification_train_processed.csv")
val_df = pd.read_csv(DATA_DIR / "classification_validation_processed.csv")

X_train = train_df.drop(columns=[TARGET_COLUMN])
y_train = train_df[TARGET_COLUMN]

X_val = val_df.drop(columns=[TARGET_COLUMN])
y_val = val_df[TARGET_COLUMN]

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("Target distribution:")
print(y_train.value_counts(normalize=True).rename("ratio"))

X_train: (65128, 291)
X_val: (16282, 291)
Target distribution:
readmitted_binary
0    0.539108
1    0.460892
Name: ratio, dtype: float64


## Feature Selection By Random Forest Importance

Random Forest can estimate feature importance. The selected feature subsets below will be used to retrain Random Forest and SVM.

In [6]:
feature_selector = RandomForestClassifier(
    n_estimators=200,
    min_samples_leaf=10,
    class_weight="balanced_subsample",
    n_jobs=-1,
    random_state=RANDOM_STATE,
)

start_time = time.time()
feature_selector.fit(X_train, y_train)
print(f"Feature selector trained in {time.time() - start_time:.2f} seconds")

feature_importance_df = pd.DataFrame({
    "feature": X_train.columns,
    "importance": feature_selector.feature_importances_,
}).sort_values("importance", ascending=False).reset_index(drop=True)

feature_importance_df.head(30)

Feature selector trained in 4.70 seconds


,feature,importance
0,num__number_inpatient,0.155965
1,num__num_medications,0.056294
2,num__num_lab_procedures,0.054407
3,num__number_diagnoses,0.047331
4,cat__discharge_disposition_id_11,0.046614
5,num__number_emergency,0.037395
6,num__time_in_hospital,0.035227
7,num__number_outpatient,0.033904
8,num__age_ordinal,0.032399
9,num__num_procedures,0.025345


In [7]:
top_k_values = [30, 50, 100, 150, X_train.shape[1]]

feature_sets = {}
for top_k in top_k_values:
    name = f"top_{top_k}" if top_k != X_train.shape[1] else "all_features"
    feature_sets[name] = feature_importance_df.head(top_k)["feature"].tolist()

{name: len(features) for name, features in feature_sets.items()}

{'top_30': 30,
 'top_50': 50,
 'top_100': 100,
 'top_150': 150,
 'all_features': 291}

## Retrain Top Models

Top models selected from baseline comparison:
- Random Forest
- SVM using LinearSVC

In [8]:
def build_models():
    return {
        "Random Forest": RandomForestClassifier(
            n_estimators=300,
            min_samples_leaf=10,
            class_weight="balanced_subsample",
            n_jobs=-1,
            random_state=RANDOM_STATE,
        ),
        "SVM": LinearSVC(
            C=1.0,
            class_weight="balanced",
            max_iter=8000,
            random_state=RANDOM_STATE,
        ),
    }


def get_score_for_auc(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, "decision_function"):
        return model.decision_function(X)
    return None


def evaluate_model(model, X_val_selected, y_val):
    y_pred = model.predict(X_val_selected)
    y_score = get_score_for_auc(model, X_val_selected)

    return {
        "accuracy": accuracy_score(y_val, y_pred),
        "precision": precision_score(y_val, y_pred, zero_division=0),
        "recall": recall_score(y_val, y_pred, zero_division=0),
        "f1_score": f1_score(y_val, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_val, y_score) if y_score is not None else None,
    }, y_pred

In [9]:
results = []
trained_models = {}
confusion_matrices = {}

for feature_set_name, selected_features in feature_sets.items():
    X_train_selected = X_train[selected_features]
    X_val_selected = X_val[selected_features]

    for model_name, model in build_models().items():
        print(f"Training {model_name} with {feature_set_name} ({len(selected_features)} features)...")
        start_time = time.time()
        model.fit(X_train_selected, y_train)
        train_time = time.time() - start_time

        metrics, y_pred = evaluate_model(model, X_val_selected, y_val)
        row = {
            "model": model_name,
            "feature_set": feature_set_name,
            "num_features": len(selected_features),
            **metrics,
            "train_time_sec": train_time,
        }
        results.append(row)

        model_key = f"{model_name}__{feature_set_name}"
        trained_models[model_key] = model
        confusion_matrices[model_key] = pd.DataFrame(
            confusion_matrix(y_val, y_pred),
            index=["actual_0", "actual_1"],
            columns=["predicted_0", "predicted_1"],
        )

        print(
            f"Done: F1={metrics['f1_score']:.4f}, "
            f"Recall={metrics['recall']:.4f}, "
            f"Precision={metrics['precision']:.4f}, "
            f"Accuracy={metrics['accuracy']:.4f}, "
            f"ROC-AUC={metrics['roc_auc']:.4f}"
        )

retrain_results_df = pd.DataFrame(results).sort_values(
    ["f1_score", "roc_auc", "accuracy"], ascending=False
).reset_index(drop=True)

retrain_results_df

Training Random Forest with top_30 (30 features)...
Done: F1=0.6061, Recall=0.6038, Precision=0.6084, Accuracy=0.6383, ROC-AUC=0.6956
Training SVM with top_30 (30 features)...
Done: F1=0.5846, Recall=0.5710, Precision=0.5988, Accuracy=0.6260, ROC-AUC=0.6785
Training Random Forest with top_50 (50 features)...
Done: F1=0.6117, Recall=0.6099, Precision=0.6135, Accuracy=0.6431, ROC-AUC=0.7011
Training SVM with top_50 (50 features)...
Done: F1=0.5969, Recall=0.5946, Precision=0.5992, Accuracy=0.6298, ROC-AUC=0.6854
Training Random Forest with top_100 (100 features)...
Done: F1=0.6133, Recall=0.6150, Precision=0.6116, Accuracy=0.6426, ROC-AUC=0.7043
Training SVM with top_100 (100 features)...
Done: F1=0.6026, Recall=0.6090, Precision=0.5964, Accuracy=0.6298, ROC-AUC=0.6889
Training Random Forest with top_150 (150 features)...
Done: F1=0.6175, Recall=0.6215, Precision=0.6135, Accuracy=0.6451, ROC-AUC=0.7058
Training SVM with top_150 (150 features)...
Done: F1=0.6088, Recall=0.6186, Precision=

,model,feature_set,num_features,accuracy,precision,recall,f1_score,roc_auc,train_time_sec
0,Random Forest,all_features,291,0.643410,0.610088,0.626999,0.618428,0.705079,12.216242
1,Random Forest,top_150,150,0.645130,0.613523,0.621535,0.617503,0.705817,6.517895
2,Random Forest,top_100,100,0.642550,0.611582,0.615005,0.613289,0.704323,5.532576
3,Random Forest,top_50,50,0.643103,0.613457,0.609941,0.611694,0.701072,4.349115
4,SVM,all_features,291,0.633767,0.598744,0.622601,0.610440,0.691045,5.670046
5,SVM,top_150,150,0.633583,0.599277,0.618603,0.608787,0.690687,3.622552
6,Random Forest,top_30,30,0.638312,0.608433,0.603811,0.606113,0.695553,3.495802
7,SVM,top_100,100,0.629837,0.596372,0.609009,0.602624,0.688893,2.759614
8,SVM,top_50,50,0.629837,0.599167,0.594616,0.596883,0.685376,1.704764
9,SVM,top_30,30,0.625967,0.598798,0.571029,0.584584,0.678465,0.811998


## Best Retrained Model

In [10]:
best_row = retrain_results_df.iloc[0]
best_model_key = f"{best_row['model']}__{best_row['feature_set']}"
best_features = feature_sets[best_row["feature_set"]]

print("Best retrained configuration:")
display(best_row.to_frame().T)

print("Best model key:", best_model_key)
print("Number of selected features:", len(best_features))

Best retrained configuration:


,model,feature_set,num_features,accuracy,precision,recall,f1_score,roc_auc,train_time_sec
0,Random Forest,all_features,291,0.64341,0.610088,0.626999,0.618428,0.705079,12.216242


Best model key: Random Forest__all_features
Number of selected features: 291


## Save Outputs

In [11]:
feature_importance_df.to_csv(OUTPUT_DIR / "feature_importance.csv", index=False)
retrain_results_df.to_csv(OUTPUT_DIR / "feature_engineering_retrain_results.csv", index=False)

pd.DataFrame({"feature": best_features}).to_csv(OUTPUT_DIR / "best_selected_features.csv", index=False)
joblib.dump(trained_models[best_model_key], OUTPUT_DIR / "best_retrained_model.joblib")

for model_key, cm in confusion_matrices.items():
    safe_name = model_key.lower().replace(" ", "_").replace("__", "_")
    cm.to_csv(OUTPUT_DIR / f"{safe_name}_validation_confusion_matrix.csv")

print(f"Saved outputs to: {OUTPUT_DIR}")

Saved outputs to: d:\HocTap\KT&XLTT\CUOIKI\train\feature_engineering\outputs
